<a href="https://colab.research.google.com/github/puteriazli/sentiment-analysis-dana-application-review/blob/main/dana_sentiment_binary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Hubungkan ke GDrive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **Install & Import Library**

In [ ]:
%pip install torch transformers emoji symspellpy scikit-learn pandas

In [ ]:
import random
import re
import emoji
import torch
import numpy as np
import pandas as pd

from collections import Counter
from symspellpy import SymSpell, Verbosity
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    cohen_kappa_score,
    confusion_matrix,
    classification_report
)
from sklearn.cluster import KMeans

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

from torch.nn import CrossEntropyLoss

In [ ]:
print("CUDA:", torch.cuda.is_available())
print("DEVICE:", DEVICE)

import transformers
print("Transformers:", transformers.__version__)

CUDA: True
DEVICE: cuda
Transformers: 5.0.0


# **Env**

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LEN = 128

USE_SUBSET = True
SUBSET_SIZE = 100_000   # FAST EXPERIMENT FIRST

# **Load Data**

In [ ]:
df = pd.read_parquet('/content/drive/MyDrive/Projek/Sentiment Analisis dan Dashboard Aplikasi DANA/data/cleaned/ulasan_dana_cleaned.parquet')

df = df.dropna(subset=["ulasan", "rating"])
df["ulasan"] = df["ulasan"].astype(str)

def rating_to_label(r):
    if r <= 2:
        return 0
    elif r == 3:
        return 1
    return 2

df["label"] = df["rating"].apply(rating_to_label)

# **Subsampling (Stratified)**

In [ ]:
if USE_SUBSET:
    df, _ = train_test_split(
        df,
        train_size=SUBSET_SIZE,
        stratify=df["label"],
        random_state=SEED
    )
    print("USING SUBSET:", df.shape)
    print(df["label"].value_counts(normalize=True))

USING SUBSET: (100000, 8)
label
2    0.70612
0    0.24720
1    0.04668
Name: proportion, dtype: float64


# **Load Slang & Colloquial**

In [ ]:
colloquial = pd.read_csv('/content/drive/MyDrive/Projek/Sentiment Analisis dan Dashboard Aplikasi DANA/data/kamus/colloquial-indonesian-lexicon.csv')
slang_dict = dict(zip(colloquial["slang"], colloquial["formal"]))

# **Build Frequency Dictionary**

In [ ]:
def tokenize(text):
    text = re.sub(r"http\S+|www\S+", "", text.lower())
    text = re.sub(r"[^a-z ]", " ", text)
    return text.split()

counter = Counter()
for text in df["ulasan"]:
    counter.update(tokenize(text))

with open("/content/drive/MyDrive/Projek/Sentiment Analisis dan Dashboard Aplikasi DANA/frequency_dictionary_id.txt", "w", encoding="utf-8") as f:
    for w, c in counter.items():
        if len(w) > 2 and c >= 3:
            f.write(f"{w} {c}\n")

# **Symspell Normalization**

In [ ]:
symspell = SymSpell(max_dictionary_edit_distance=2)
symspell.load_dictionary("/content/drive/MyDrive/Projek/Sentiment Analisis dan Dashboard Aplikasi DANA/frequency_dictionary_id.txt", 0, 1)

def normalize_elongation(w):
    return re.sub(r"(.)\1{2,}", r"\1", w)

def normalize_word(w):
    if w in slang_dict:
        return slang_dict[w]
    w = normalize_elongation(w)
    sug = symspell.lookup(w, Verbosity.CLOSEST, max_edit_distance=2)
    return sug[0].term if sug else w

def clean_text(text):
    text = emoji.demojize(text.lower(), delimiters=(" ", " "))
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z_ ]", " ", text)
    return " ".join(normalize_word(t) for t in text.split())

df["clean_text"] = df["ulasan"].apply(clean_text)

# **Split Data**

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df["label"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label"],
    random_state=SEED
)

# **Tokenisasi**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=MAX_LEN
        )
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_ds = SentimentDataset(train_df["clean_text"].tolist(), train_df["label"].tolist())
val_ds   = SentimentDataset(val_df["clean_text"].tolist(),   val_df["label"].tolist())
test_ds  = SentimentDataset(test_df["clean_text"].tolist(),  test_df["label"].tolist())

# **Focal Loss**

In [ ]:
class FocalLoss(torch.nn.Module):
    def __init__(self, alpha, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ce = torch.nn.CrossEntropyLoss(reduction="none")

    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        pt = torch.exp(-ce_loss)
        focal = self.alpha[targets] * (1 - pt) ** self.gamma * ce_loss
        return focal.mean()

# **Model**

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
).to(DEVICE)

class_counts = train_df["label"].value_counts().sort_index()
alpha = (1.0 / class_counts).values
alpha = torch.tensor(alpha / alpha.sum(), dtype=torch.float).to(DEVICE)

model.loss_fct = FocalLoss(alpha=alpha, gamma=2.0)

# freeze all except last encoder
for name, param in model.base_model.named_parameters():
    if not name.startswith("encoder.layer.11"):
        param.requires_grad = False

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# **Training**

In [ ]:
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {
        "accuracy": accuracy_score(p.label_ids, preds),
        "macro_f1": f1_score(p.label_ids, preds, average="macro"),
        "cohen_kappa": cohen_kappa_score(p.label_ids, preds)
    }

args = TrainingArguments(
    output_dir="./model_dana_focal",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Cohen Kappa
1,0.382832,0.368110,0.875067,0.579541,0.707345
2,0.373752,0.366170,0.877400,0.580197,0.702667
3,0.374982,0.361085,0.878800,0.582213,0.710889
4,0.360387,0.361534,0.879000,0.582309,0.710521


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Cohen Kappa
1,0.382832,0.368110,0.875067,0.579541,0.707345
2,0.373752,0.366170,0.877400,0.580197,0.702667
3,0.374982,0.361085,0.878800,0.582213,0.710889
4,0.360387,0.361534,0.879000,0.582309,0.710521
5,0.371571,0.360368,0.879800,0.583132,0.713588


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=10940, training_loss=0.3743814192916619, metrics={'train_runtime': 970.4718, 'train_samples_per_second': 360.649, 'train_steps_per_second': 11.273, 'total_flos': 2.30224240512e+16, 'train_loss': 0.3743814192916619, 'epoch': 5.0})

# **Evaluasi + Eror Analysis**

In [ ]:
test_preds = trainer.predict(test_ds)
y_true = test_df["label"].values
y_pred = np.argmax(test_preds.predictions, axis=1)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, digits=4))

[[ 3210     0   498]
 [  336     1   363]
 [  574     0 10018]]
              precision    recall  f1-score   support

           0     0.7791    0.8657    0.8201      3708
           1     1.0000    0.0014    0.0029       700
           2     0.9209    0.9458    0.9332     10592

    accuracy                         0.8819     15000
   macro avg     0.9000    0.6043    0.5854     15000
weighted avg     0.8895    0.8819    0.8618     15000



,ulasan,label,pred
298504,Mantap pokeee,0,2
733193,Min dana saya kok membal ga bisa Dibuka tolong...,0,2
200486,knapa tidak ada dana cicil nya perbaiki dong,0,2
681295,Kenapa daba saya gg bisa di bukaya,0,2
352701,min kenapa vitur indihome speedy di dana gk ad...,0,2
255819,bravo,0,2
134398,dana cicil tidak muncul😔,0,2
506850,Kenapa di akun dana saya tidak ada dana cicil ...,0,2
114681,"kartu nomor akun dana saya telah hilang, sebel...",0,2
458004,Udh update tpi dana cicilnya gk muncul,0,2


In [ ]:
# dump negative errors
error_df = test_df.copy()
error_df["pred"] = y_pred
errors_neg = error_df[(error_df.label == 0) & (error_df.pred != 0)]
errors_neg.sample(20)[["ulasan", "label", "pred"]]

# **Inference**

In [ ]:
def predict_sentiment(text):
    text = clean_text(text)
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN
    ).to(DEVICE)

    with torch.no_grad():
        pred = torch.argmax(model(**inputs).logits, dim=1).item()

    return {0: "negative", 1: "neutral", 2: "positive"}[pred]

In [ ]:
SAVE_DIR = "model_sentiment_final"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('model_sentiment_final/tokenizer_config.json',
 'model_sentiment_final/tokenizer.json')

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

MODEL_PATH = "./model_sentiment_final"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()  # PENTING

label_map = {0: "negative", 1: "neutral", 2: "positive"}

def predict_sentiment(text):
    inputs = tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=128,
        return_tensors="pt"
    )

    with torch.no_grad():  # PENTING
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()

    return {
        "label": label_map[pred],
        "confidence": float(probs[0][pred])
    }

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
# contoh
predict_sentiment("apk lemot")

{'label': 'negative', 'confidence': 0.9209776520729065}